# Practical 2: Sentiment Classification using DNN — IMDB Dataset

## A. Dataset Description
| Field | Details |
|---|---|
| **Dataset Name** | IMDB Movie Reviews |
| **Source** | `keras.datasets.imdb` |
| **Total Samples** | 50,000 (combined train + test) |
| **Train / Test** | 40,000 (80%) / 10,000 (20%) |
| **Input Features** | Binary BoW vector of top 10,000 most frequent words |
| **Output Label** | 1 = Positive, 0 = Negative |

## B. Model Architecture
| Layer | Type | Units | Activation | Dropout |
|---|---|---|---|---|
| 1 | Dense (Input) | 50 | ReLU | 0.3 |
| 2 | Dense (Hidden) | 50 | ReLU | 0.2 |
| 3 | Dense (Hidden) | 50 | ReLU | — |
| 4 | Dense (Output) | 1 | Sigmoid | — |

**Type:** DNN (Deep Neural Network)

## C. Hyperparameters
| Hyperparameter | Value |
|---|---|
| Learning Rate | Adam default (1e-3) |
| Batch Size | 500 |
| Epochs | 20 (EarlyStopping, patience=3) |
| Optimizer | Adam |
| Loss Function | Binary Crossentropy |

## D. Training Details
- **Regularization:** Dropout(0.3) after Layer 1, Dropout(0.2) after Layer 2
- **Early Stopping:** monitors `val_loss`, patience=3, restores best weights


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from keras.datasets import imdb
from keras import models, layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc)
import tensorflow as tf

# ── Load Dataset ───────────────────────────────────────
print("Loading IMDB dataset...")
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=10000)

data   = np.concatenate((X_train, X_test), axis=0)
labels = np.concatenate((y_train, y_test), axis=0)
print(f"Total samples: {len(data)}")

# ── Vectorization (Binary BoW) ─────────────────────────
def vectorize(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

data   = vectorize(data)
labels = np.array(labels).astype("float32")
print(f"Feature shape: {data.shape}")


ModuleNotFoundError: No module named 'numpy'

In [ ]:
# ── Train-Test Split (80-20) ───────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")


In [ ]:
# ── Model Architecture ────────────────────────────────
model = models.Sequential([
    layers.Dense(50, activation="relu", input_shape=(10000,)),
    layers.Dropout(0.3),
    layers.Dense(50, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(50, activation="relu"),
    layers.Dense(1,  activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
# ── Training ──────────────────────────────────────────
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=20, batch_size=500,
    validation_data=(X_test, y_test),
    callbacks=[early_stop], verbose=1
)
print(f"\nStopped at epoch {len(history.history['loss'])}")


In [ ]:
# ── Evaluation & E. Performance Metrics ───────────────
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
y_prob = model.predict(X_test).flatten()
y_pred = (y_prob > 0.5).astype("int32")

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
acc       = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("=" * 50)
print("E. PERFORMANCE METRICS")
print("=" * 50)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))


In [ ]:
# ── Confusion Matrix ──────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Negative","Positive"],
            yticklabels=["Negative","Positive"])
plt.title("Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout(); plt.show()

# ── ROC Curve ─────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1],[0,1],'navy--'); plt.title("ROC Curve")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()
print(f"AUC Score: {roc_auc:.4f}")


## F. Training Behavior

In [ ]:
# ── F. Accuracy vs Epoch ──────────────────────────────
plt.figure(figsize=(9, 4))
plt.plot(history.history['accuracy'],     label='Train', color='steelblue')
plt.plot(history.history['val_accuracy'], label='Val',   color='orange')
plt.title("Accuracy vs Epoch"); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# ── F. Loss vs Epoch ──────────────────────────────────
plt.figure(figsize=(9, 4))
plt.plot(history.history['loss'],     label='Train', color='steelblue')
plt.plot(history.history['val_loss'], label='Val',   color='orange')
plt.title("Loss vs Epoch"); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


## G. Sample Predictions

In [ ]:
# ── G. Sample Predictions ─────────────────────────────
label_names = {0: "Negative", 1: "Positive"}
print(f"{'#':<5} {'Actual':>12} {'Predicted':>12} {'Confidence':>12}")
print("-" * 45)
for i in range(10):
    actual    = int(y_test[i])
    predicted = int(y_pred[i])
    conf      = y_prob[i] if predicted == 1 else 1 - y_prob[i]
    print(f"{i+1:<5} {label_names[actual]:>12} {label_names[predicted]:>12} {conf:>11.2%}")


## H. Observations / Analysis

1. **Overfitting/Underfitting:**
   - Dropout (0.3, 0.2) and EarlyStopping together prevent overfitting effectively.
   - If val_loss rises while train_loss falls, reduce model capacity or increase Dropout.

2. **Hyperparameter Effects:**
   - Batch size 500 speeds training but may slightly reduce generalisation.
   - Smaller batches (128–256) can improve validation accuracy marginally.

3. **Anomalies:**
   - Binary BoW ignores word order — sarcasm and negation may be misclassified.
   - AUC > 0.95 indicates excellent class separation on this dataset.
